In [51]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer
import cv2
import pandas as pd
import numpy as np
import json
from shapely.geometry import Polygon
from sklearn.cluster import DBSCAN

# ======================== Drawing ========================
# def draw_annotations_on_image(full_image_array, slices, object_infos, output_path):
#     img = np.moveaxis(full_image_array[:3], 0, -1)
#     img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
#     img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
#     overlay = img.copy()

#     slice_offsets = {tuple(slice_data["offset_xy"]): slice_data for slice_data in slices}

#     for obj in object_infos:
#         try:
#             lon = float(obj['longitude'])
#             lat = float(obj['latitude'])
#             for slice_data in slices:
#                 transform = slice_data['transform']
#                 row, col = rasterio.transform.rowcol(transform, lon, lat)
#                 pixel_x = col + slice_data["offset_xy"][0]
#                 pixel_y = row + slice_data["offset_xy"][1]
#                 break

#             # Bounding box overlay
#             x1_global = int(obj['global_x'] - obj['bbox_width'] / 2)
#             y1_global = int(obj['global_y'] - obj['bbox_height'] / 2)
#             x2_global = int(obj['global_x'] + obj['bbox_width'] / 2)
#             y2_global = int(obj['global_y'] + obj['bbox_height'] / 2)
#             label = f"Cls:{obj['class']} ({obj['confidence']:.2f})"
#             cv2.rectangle(overlay, (x1_global, y1_global), (x2_global, y2_global), (0, 255, 0), 3)
#             cv2.putText(overlay, label, (x1_global, y1_global - 10), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (255, 255, 255), 5)
#             # Mask overlay
#             mask_points = np.array(json.loads(obj['mask_points']), dtype=np.int32)
#             mask_points_global = mask_points + np.array([int(obj['global_x'] - obj['x_center']),
#                                                          int(obj['global_y'] - obj['y_center'])])
#             if len(mask_points_global) >= 3:
#                 mask_img = np.zeros_like(img)
#                 cv2.fillPoly(mask_img, [mask_points_global], (0, 0, 255))
#                 overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.3, 0)

#             cv2.circle(overlay, (int(pixel_x), int(pixel_y)), 8, (0, 0, 255), -1)
#         except:
#             continue

#     output_path_jpg = output_path.replace(".png", ".jpg")
#     cv2.imwrite(output_path_jpg, overlay)
#     print(f"📸 Annotated image saved to: {output_path_jpg}")

def draw_annotations_on_image(full_image_array, slices, object_infos, output_path):
    """
    Overlays bounding boxes, masks, and lat/lon centers with:
    - Transparent masks (alpha=0.3)
    - Class/conf labels
    - Red dot + lat/lon text
    - JPEG output
    """
    import cv2
    import numpy as np
    import json
    from rasterio.transform import rowcol

    # Convert rasterio (C,H,W) to OpenCV (H,W,C)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    overlay = img.copy()

    # Offset lookup
    slice_offsets = {tuple(slice_data["offset_xy"]): slice_data for slice_data in slices}

    for obj in object_infos:
        tile_name = obj['tile_name']
        offset_xy = (int(obj['global_x'] - obj['x_center']), int(obj['global_y'] - obj['y_center']))

        # Bounding box
        x1_global = int(obj['bbox_x1'] + offset_xy[0])
        y1_global = int(obj['bbox_y1'] + offset_xy[1])
        x2_global = int(obj['bbox_x2'] + offset_xy[0])
        y2_global = int(obj['bbox_y2'] + offset_xy[1])

        conf = obj['confidence']
        cls = obj['class']
        label = f"Cls:{cls} ({conf:.2f})"

        # Draw bounding box
        cv2.rectangle(overlay, (x1_global, y1_global), (x2_global, y2_global), (0, 255, 0), 3)

        # Label text
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 2.5
        thickness = 5
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
        text_origin = (x1_global, max(y1_global - 10, text_size[1]))
        cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

        # Mask overlay
        try:
            mask_points = np.array(json.loads(obj['mask_points']), dtype=np.int32)
            mask_points_global = mask_points + np.array([offset_xy[0], offset_xy[1]])

            if len(mask_points_global) >= 3:
                mask_img = np.zeros_like(img)
                cv2.fillPoly(mask_img, [mask_points_global], (0, 0, 255))
                cv2.polylines(mask_img, [mask_points_global], isClosed=True, color=(0, 0, 255), thickness=2)
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.7, 0)
        except:
            pass

        # Lat/Lon center overlay
        try:
            lat = float(obj['latitude'])
            lon = float(obj['longitude'])

            # Use reverse transform to get row,col (y,x)
            for slice_data in slices:
                transform = slice_data['transform']
                try:
                    row, col = rowcol(transform, lon, lat)
                    pixel_x = col + slice_data["offset_xy"][0]
                    pixel_y = row + slice_data["offset_xy"][1]
                    break
                except:
                    continue

            # Draw center point
            cv2.circle(overlay, (int(pixel_x), int(pixel_y)), 8, (0, 0, 255), -1)

            # Add lat/lon text
            ll_label = f"({lat:.5f}, {lon:.5f})"
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 3)
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        except:
            continue

    # Save as JPEG
    output_path_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(output_path_jpg, overlay)
    print(f"📸 Annotated image saved to: {output_path_jpg}")

# ======================== Sliding ========================
from rasterio.windows import Window
from rasterio.transform import from_origin

def slide_geotiff(geotiff_path, window_size, stride):
    """
    Splits a GeoTIFF into window_size×window_size tiles with given stride,
    **and** adds extra overlapping tiles on the right/bottom so that no pixel is left out.
    """
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width

        # 1) Compute all x-starts
        if width <= window_size:
            x_starts = [0]
        else:
            x_starts = list(range(0, width - window_size + 1, stride))
            # leftover on right?
            if x_starts[-1] + window_size < width:
                x_starts.append(width - window_size)

        # 2) Compute all y-starts
        if height <= window_size:
            y_starts = [0]
        else:
            y_starts = list(range(0, height - window_size + 1, stride))
            # leftover on bottom?
            if y_starts[-1] + window_size < height:
                y_starts.append(height - window_size)

        # 3) Build tiles for every (x,y) pair
        for y in y_starts:
            for x in x_starts:
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": src.crs,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })

    return tiles


# ======================== Per-Tile Suppression ========================
from torchvision.ops import nms
def maximum_likelihood_suppression(results, conf_threshold=0.5, iou_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results
    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    keep = scores >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results
    keep_indices = nms(boxes[keep], scores[keep], iou_threshold)
    indices_to_keep = torch.nonzero(keep).squeeze(1)[keep_indices]
    results.boxes.data = results.boxes.data[indices_to_keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[indices_to_keep]
    return results

# ======================== Per-Tile Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out


# ======================== Full Pipeline Runner ========================
def run_inference(folder_path, model_path, final_csv_path, window_size=3000, stride=3000, batch_size=16):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    # GPU Warmup
    _ = model([Image.new('RGB', (window_size, window_size))], device=device, verbose=False)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos = []
    for tif_image in tqdm(tif_images, desc="Overall Progress"):
        slices = slide_geotiff(tif_image, window_size, stride)
        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                accumulate_object_infos.extend(object_info)

    pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Final results saved to {final_csv_path}")
    # print("\n🔍 Applying Global Suppression...")
    # suppressed_infos = global_suppression_pipeline(accumulate_object_infos)
    # pd.DataFrame(suppressed_infos).to_csv(final_csv_path, index=False)
    # print(f"✅ Final suppressed results saved to {final_csv_path}")

    # # Visualization after suppression
    # for tif_image in tif_images:
    #     with rasterio.open(tif_image) as src:
    #         full_image_array = src.read()
    #     image_name = os.path.splitext(os.path.basename(tif_image))[0]
    #     draw_annotations_on_image(full_image_array, slide_geotiff(tif_image, window_size, stride),
    #                               [obj for obj in suppressed_infos if obj['tile_name'].split(',')[0] == os.path.basename(tif_image)],
    #                               f"annotated_output_test/{image_name}_annotated.png")



# ======================== Visualization ========================
def run_viz_sup(folder_path, csv_sup="annotated_output_test/final_sup.csv", window_size=3000, stride=3000):
    suppressed_infos = pd.read_csv(csv_sup)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    for tif_image in tif_images:
        with rasterio.open(tif_image) as src:
            full_image_array = src.read()
        image_name = os.path.splitext(os.path.basename(tif_image))[0]

        # Correct way to filter using DataFrame
        matched_objs = suppressed_infos[suppressed_infos['tile_name'].str.split(',').str[0] == os.path.basename(tif_image)]
        # Convert filtered DataFrame to dict list
        matched_objs = matched_objs.to_dict(orient='records')

        draw_annotations_on_image(full_image_array, slide_geotiff(tif_image, window_size, stride),
                                  matched_objs,
                                  f"annotated_output_test_1/{image_name}_annotated.jpg")

# ======================== making ractangles ========================
import pandas as pd
import json
from shapely.geometry import Polygon

def make_rectangles(input_csv: str, output_csv: str):
    """
    Reads input_csv (with a 'mask_points_gis' column of JSON [[x,y],…]),
    computes an axis-aligned bounding rectangle for each mask,
    and writes out all original columns plus:

      - rect_mask_points: JSON list of the 4 rectangle corners [[x1,y1],…]
      - rect_bbox:        (minx, miny, maxx, maxy)
      - rect_area:        area of that rectangle
      - rect_x_center:    rectangle centroid x
      - rect_y_center:    rectangle centroid y

    to output_csv.
    """
    df = pd.read_csv(input_csv)

    # Prepare storage for new columns
    rect_points = []
    rect_bboxes = []
    rect_areas = []
    rect_centers = []

    for raw in df['mask_points_gis']:
        pts = json.loads(raw)
        poly = Polygon(pts)

        # 1) Axis-aligned bounding box
        minx, miny, maxx, maxy = poly.bounds

        # 4 corner coordinates, in order
        rect = poly.minimum_rotated_rectangle
        # shapely may return 5 points (first == last), so pick the unique 4:
        coords = list(rect.exterior.coords)[:-1]
        corners = [[x, y] for x, y in coords]


        # collect
        rect_points.append(json.dumps(corners))
        rect_bboxes.append((minx, miny, maxx, maxy))
        rect_areas.append(rect.area)
        cx, cy = rect.centroid.x, rect.centroid.y
        rect_centers.append((cx, cy))

    # assign new columns
    df['rect_mask_points'] = rect_points
    df['rect_bbox']         = rect_bboxes
    df['rect_area']         = rect_areas
    df['rect_x_center']     = [c[0] for c in rect_centers]
    df['rect_y_center']     = [c[1] for c in rect_centers]

    # save
    df.to_csv(output_csv, index=False)
    print(f"Written {len(df)} rows with rectangle info to {output_csv!r}")

# make_rectangles(input_csv="annotated_output_test/final.csv", output_csv="annotated_output_test/final_rect.csv")


# ======================== Global Suppression ========================
import pandas as pd
import ast
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
from shapely.validation import make_valid

def merge_adjacent_polygons(input_csv: str, output_csv: str, eps: float = 0.0):
    df = pd.read_csv(input_csv)

    def parse_and_clean(s):
        pts = ast.literal_eval(s)
        poly = Polygon(pts)
        return make_valid(poly) if not poly.is_valid else poly

    df['__poly'] = df['mask_points_gis'].apply(parse_and_clean)
    unvisited = set(df.index)
    merged_rows = []

    while unvisited:
        seed = unvisited.pop()
        cluster = {seed}
        queue = [seed]
        while queue:
            idx = queue.pop()
            poly = df.at[idx, '__poly']
            buf = poly.buffer(eps)
            neighbors = [j for j in list(unvisited) if buf.intersects(df.at[j, '__poly'])]
            for j in neighbors:
                unvisited.remove(j)
                cluster.add(j)
                queue.append(j)

        polys = [df.at[i, '__poly'] for i in cluster]
        merged = unary_union(polys)
        # ensure validity
        if not merged.is_valid:
            merged = merged.buffer(0)

        # ───── Normalize to a single Polygon ─────
        if isinstance(merged, GeometryCollection):
            parts = [g for g in merged.geoms if isinstance(g, (Polygon, MultiPolygon))]
            merged = unary_union(parts)
        if isinstance(merged, MultiPolygon):
            merged = max(merged.geoms, key=lambda p: p.area)
        if not isinstance(merged, Polygon):
            merged = merged.convex_hull

        # ───── Extract exterior coords ─────
        coords = list(merged.exterior.coords)

        # pick highest-confidence row
        best = (
            df.loc[list(cluster)]
              .sort_values('confidence', ascending=False)
              .iloc[0]
              .copy()
        )

        best['mask_points_gis'] = repr(coords)
        minx, miny, maxx, maxy = merged.bounds
        best['bbox']     = [minx, miny, maxx, maxy]
        best['x_center'] = (minx + maxx) / 2
        best['y_center'] = (miny + maxy) / 2
        best['area']     = merged.area

        merged_rows.append(best)

    out = pd.DataFrame(merged_rows).drop(columns='__poly', errors='ignore')
    out.to_csv(output_csv, index=False)
    print(f"Merged {len(df)} → {len(out)} detections; saved to {output_csv}")


# ======================== Visualization ========================
def apply_global_sup(input_csv="annotated_output_test_1/final.csv", output_csv="annotated_output_test_1/final_sup.csv"):
    # accumulate_object_infos = pd.read_csv(input_csv)
    print("\n🔍 Applying Global Suppression...")
    merge_adjacent_polygons(input_csv="annotated_output_test_1/final.csv", 
                            output_csv="annotated_output_test_1/final_sup.csv")
    # pd.DataFrame(suppressed_infos).to_csv(output_csv, index=False)
    print(f"✅ Final suppressed results saved to {output_csv}")


In [52]:
# # folder_path="D:/Data/al_30cm_2021_transfer_1_of_2/31085",
# # folder_path="C:/Users/mza0288/Documents/Code/geotiff_sample",
run_inference(
    folder_path="C:/Users/mza0288/Documents/Code/test_sample",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="annotated_output_test_1/final.csv",
    window_size=3000, stride=3000, batch_size=62
)

Using device: cuda


Overall Progress: 100%|████████████████████████████████████████████████████████████████| 11/11 [06:40<00:00, 36.43s/it]

✅ Final results saved to annotated_output_test_1/final.csv


In [53]:
apply_global_sup()
# run_viz_sup(folder_path="C:/Users/mza0288/Documents/Code/test_sample", csv_sup="annotated_output_test/final.csv")
# run_viz_sup(folder_path="C:/Users/mza0288/Documents/Code/test_sample")
make_rectangles(input_csv="annotated_output_test_1/final.csv", output_csv="annotated_output_test_1/final_rect.csv")


🔍 Applying Global Suppression...
Merged 182 → 115 detections; saved to annotated_output_test_1/final_sup.csv
✅ Final suppressed results saved to annotated_output_test_1/final_sup.csv
Written 182 rows with rectangle info to 'annotated_output_test_1/final_rect.csv'


In [8]:
import os
import glob
import json
import ast
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.windows import Window
from rasterio.transform import from_origin

def draw_annotations_with_rectangles(full_image_array, slices, object_infos, output_path):
    """
    Overlays:
      • Red 5px boundaries around each sliding window
      • YOLO bounding boxes (green)
      • YOLO masks (transparent red fill + outline)
      • Axis-aligned rectangle from 'rect_mask_points' (blue outline)
      • Rectangle centroid (blue dot)
      • Rectangle area & bbox text (blue)
      • Class/conf labels
      • JPEG output
    """
    # 1) Prepare base image (BGR)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    overlay = img.copy()

    # 2) Draw sliding-window boundaries in red
    for sl in slices:
        x_off, y_off = sl['offset_xy']
        h = sl['img_array'].shape[1]  # window height
        w = sl['img_array'].shape[2]  # window width
        cv2.rectangle(overlay,
                      (x_off, y_off),
                      (x_off + w, y_off + h),
                      (0, 0, 255),  # red
                      5)            # thickness

    # 3) Styles for later
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs_label, th_label = 2.5, 5
    fs_area,  th_area  = 1.5, 3
    fs_box,   th_box   = 1.2, 2

    # 4) Draw each object’s annotations
    for obj in object_infos:
        # compute tile→full-image offset
        ox = int(obj['global_x'] - obj['x_center'])
        oy = int(obj['global_y'] - obj['y_center'])
        offset = np.array([ox, oy], dtype=np.int32)

        # # — YOLO bbox & label —
        # x1, y1 = int(obj['bbox_x1'] + ox), int(obj['bbox_y1'] + oy)
        # x2, y2 = int(obj['bbox_x2'] + ox), int(obj['bbox_y2'] + oy)
        # cv2.rectangle(overlay, (x1, y1), (x2, y2), (0,255,0), 3)
        # label = f"Cls:{obj['class']} ({obj['confidence']:.2f})"
        # ts = cv2.getTextSize(label, font, fs_label, th_label)[0]
        # origin = (x1, max(y1 - 10, ts[1]))
        # cv2.putText(overlay, label, origin, font, fs_label, (0,0,0), th_label+2)
        # cv2.putText(overlay, label, origin, font, fs_label, (255,255,255), th_label)

        # # — YOLO mask —
        # try:
        #     pts = np.array(json.loads(obj['mask_points']), dtype=np.int32) + offset
        #     mask_img = np.zeros_like(img)
        #     cv2.fillPoly(mask_img, [pts], (0,0,255))
        #     cv2.polylines(mask_img, [pts], True, (0,0,255), 2)
        #     overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.3, 0)
        # except:
        #     pass

        # — Axis-aligned rectangle & info —
        try:
            # corners in GIS coords → pixel coords
            rect_gis = json.loads(obj['rect_mask_points'])
            pix_pts = []
            for X, Y in rect_gis:
                for sl in slices:
                    try:
                        r, c = rowcol(sl['transform'], X, Y)
                        px = c + sl['offset_xy'][0]
                        py = r + sl['offset_xy'][1]
                        pix_pts.append([px, py])
                        break
                    except:
                        continue
            pix_pts = np.array(pix_pts, dtype=np.int32)

            # draw rectangle outline
            cv2.polylines(overlay, [pix_pts], True, (255,0,0), 3)

            # draw centroid
            Xc, Yc = float(obj['rect_x_center']), float(obj['rect_y_center'])
            for sl in slices:
                try:
                    rr, cc = rowcol(sl['transform'], Xc, Yc)
                    cx = int(cc + sl['offset_xy'][0])
                    cy = int(rr + sl['offset_xy'][1])
                    break
                except:
                    continue
            cv2.circle(overlay, (cx, cy), 6, (255,0,0), -1)

            # annotate area
            area_txt = f"A:{obj['rect_area']:.1f}"
            pt_area = (pix_pts[0][0], pix_pts[0][1] - 10)
            cv2.putText(overlay, area_txt, pt_area, font, fs_area, (255,0,0), th_area)

            # annotate bbox tuple
            bbox = ast.literal_eval(obj['rect_bbox'])
            bbox_txt = f"BB:{bbox[0]:.0f},{bbox[1]:.0f},{bbox[2]:.0f},{bbox[3]:.0f}"
            br = pix_pts[2]
            pt_box = (br[0] - 200, br[1] + 30)
            cv2.putText(overlay, bbox_txt, pt_box, font, fs_box, (255,0,0), th_box)

        except:
            pass

    # 5) Save as JPEG
    out_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(out_jpg, overlay)
    print(f"📸 Saved annotated image with rectangles and window grid to: {out_jpg}")


def run_viz_rect(folder_path, csv_sup="annotated_output_test/final_rect.csv",
                 window_size=3000, stride=3000):
    suppressed_infos = pd.read_csv(csv_sup)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    for tif_image in tif_images:
        with rasterio.open(tif_image) as src:
            full_image_array = src.read()
        image_name = os.path.splitext(os.path.basename(tif_image))[0]

        matched = suppressed_infos[
            suppressed_infos['tile_name'].str.split(',').str[0] == os.path.basename(tif_image)
        ]
        objs = matched.to_dict(orient='records')

        draw_annotations_with_rectangles(
            full_image_array,
            slide_geotiff(tif_image, window_size, stride),
            objs,
            f"annotated_output_test_1/{image_name}_rect_annotated.jpg"
        )


# run_viz_rect(folder_path="C:/Users/mza0288/Documents/Code/test_sample", csv_sup="annotated_output_test_1/final_rect_merged.csv")

In [54]:
import pandas as pd
import ast
from shapely.geometry import Polygon, LineString
from shapely.ops import unary_union
import itertools

def polygons_are_adjacent(poly1, poly2, threshold):
    edges1 = [LineString([poly1.exterior.coords[i], poly1.exterior.coords[i+1]]) 
              for i in range(len(poly1.exterior.coords)-1)]
    edges2 = [LineString([poly2.exterior.coords[i], poly2.exterior.coords[i+1]]) 
              for i in range(len(poly2.exterior.coords)-1)]
    
    # Check each edge against edges of another polygon
    for edge1, edge2 in itertools.product(edges1, edges2):
        if edge1.distance(edge2) <= threshold:
            return True
    return False

def merge_adjacent_masks(input_csv, output_csv, distance_threshold=2.0):
    df = pd.read_csv(input_csv)
    df['polygon'] = df['rect_mask_points'].apply(lambda x: Polygon(ast.literal_eval(x)))
    polygons = df['polygon'].tolist()

    n = len(polygons)
    adjacency_list = {i: set() for i in range(n)}

    # Build adjacency list carefully based on edge proximity
    for i, j in itertools.combinations(range(n), 2):
        if polygons_are_adjacent(polygons[i], polygons[j], distance_threshold):
            adjacency_list[i].add(j)
            adjacency_list[j].add(i)

    # Group polygons using adjacency list
    visited = set()
    merged_groups = []

    def dfs(current_idx, group):
        visited.add(current_idx)
        group.add(current_idx)
        for neighbor in adjacency_list[current_idx]:
            if neighbor not in visited:
                dfs(neighbor, group)

    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            merged_groups.append(group)

    merged_data = []

    for group_indices in merged_groups:
        merge_polys = [polygons[i] for i in group_indices]
        merged_geometry = unary_union(merge_polys)

        if merged_geometry.geom_type == 'MultiPolygon':
            merged_geometry = list(merged_geometry.geoms)
        else:
            merged_geometry = [merged_geometry]

        for single_geom in merged_geometry:
            group_rows = df.loc[list(group_indices)]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()

            minx, miny, maxx, maxy = single_geom.bounds
            highest_confidence_row['mask_points_gis'] = str(list(single_geom.exterior.coords))
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            highest_confidence_row['rect_area'] = single_geom.area
            centroid = single_geom.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny

            merged_data.append(highest_confidence_row)

    final_df = pd.DataFrame(merged_data)

    final_df.drop(columns=['polygon'], inplace=True)

    final_df.to_csv(output_csv, index=False)

# Example usage:
# merge_adjacent_masks('final_rect.csv', 'final_rect_merged.csv', distance_threshold=2.0)


In [55]:
# Example usage
merge_adjacent_masks('annotated_output_test_1/final_rect.csv', 'annotated_output_test_1/final_rect_merged.csv', distance_threshold=1.0)

In [58]:
run_viz_rect(folder_path="C:/Users/mza0288/Documents/Code/test_sample", csv_sup="annotated_output_test_1/final_rect_merged.csv", window_size=3000, stride=3000)

📸 Saved annotated image with rectangles and window grid to: annotated_output_test_1/m_3108501_ne_16_030_20211103_rect_annotated.jpg
📸 Saved annotated image with rectangles and window grid to: annotated_output_test_1/m_3108501_nw_16_030_20211103_rect_annotated.jpg
📸 Saved annotated image with rectangles and window grid to: annotated_output_test_1/m_3108501_se_16_030_20211103_rect_annotated.jpg


KeyboardInterrupt: 

In [57]:
import pandas as pd
import ast
from shapely.geometry import Polygon, LineString
from shapely.ops import unary_union
import itertools

def polygons_are_adjacent(poly1, poly2, threshold):
    edges1 = [LineString([poly1.exterior.coords[i], poly1.exterior.coords[i+1]]) 
              for i in range(len(poly1.exterior.coords)-1)]
    edges2 = [LineString([poly2.exterior.coords[i], poly2.exterior.coords[i+1]]) 
              for i in range(len(poly2.exterior.coords)-1)]
    
    for edge1, edge2 in itertools.product(edges1, edges2):
        if edge1.distance(edge2) <= threshold:
            return True
    return False

def merge_adjacent_masks_debug(input_csv, output_csv, distance_threshold=2.0):
    df = pd.read_csv(input_csv)
    df['polygon'] = df['rect_mask_points'].apply(lambda x: Polygon(ast.literal_eval(x)))
    polygons = df['polygon'].tolist()

    n = len(polygons)
    adjacency_list = {i: set() for i in range(n)}

    print("Building adjacency list...")
    for i, j in itertools.combinations(range(n), 2):
        adjacent = polygons_are_adjacent(polygons[i], polygons[j], distance_threshold)
        if adjacent:
            adjacency_list[i].add(j)
            adjacency_list[j].add(i)
            print(f"Polygons {i} and {j} are adjacent.")

    visited = set()
    merged_groups = []

    def dfs(current_idx, group):
        visited.add(current_idx)
        group.add(current_idx)
        for neighbor in adjacency_list[current_idx]:
            if neighbor not in visited:
                dfs(neighbor, group)

    print("Identifying merged groups...")
    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            print(f"Merged group found: {group}")
            merged_groups.append(group)

    merged_data = []

    for group_indices in merged_groups:
        merge_polys = [polygons[i] for i in group_indices]
        merged_geometry = unary_union(merge_polys)

        if merged_geometry.geom_type == 'MultiPolygon':
            print(f"Group {group_indices} resulted in MultiPolygon with {len(merged_geometry.geoms)} polygons.")
            merged_geometry = list(merged_geometry.geoms)
        else:
            merged_geometry = [merged_geometry]

        for idx, single_geom in enumerate(merged_geometry):
            group_rows = df.loc[list(group_indices)]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()

            minx, miny, maxx, maxy = single_geom.bounds
            highest_confidence_row['mask_points_gis'] = str(list(single_geom.exterior.coords))
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            highest_confidence_row['rect_area'] = single_geom.area
            centroid = single_geom.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny

            merged_data.append(highest_confidence_row)

            print(f"Group {group_indices}, polygon {idx} merged, area: {single_geom.area:.2f}")

    final_df = pd.DataFrame(merged_data)

    final_df.drop(columns=['polygon'], inplace=True)

    final_df.to_csv(output_csv, index=False)
    print(f"Final merged CSV saved to {output_csv}")

# Example usage
merge_adjacent_masks('annotated_output_test_1/final_rect.csv', 'annotated_output_test_1/final_rect_merged.csv', distance_threshold=5.0)


In [46]:
import pandas as pd
import ast
from typing import Tuple

def _parse_bbox(s: str) -> Tuple[float, float, float, float]:
    """Turn a string '(x1, y1, x2, y2)' into a tuple of floats."""
    return tuple(ast.literal_eval(s))

def _bbox_iou(b1, b2) -> float:
    """Compute IoU between two AABB boxes."""
    x1, y1, x2, y2 = b1
    x1p, y1p, x2p, y2p = b2
    xx1, yy1 = max(x1, x1p), max(y1, y1p)
    xx2, yy2 = min(x2, x2p), min(y2, y2p)
    w, h = max(0, xx2 - xx1), max(0, yy2 - yy1)
    inter = w * h
    area1 = max(0, x2 - x1) * max(0, y2 - y1)
    area2 = max(0, x2p - x1p) * max(0, y2p - y1p)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def suppress_overlapping_bboxes(
    input_csv: str,
    output_csv: str,
    iou_thresh: float = 0.9
):
    """
    Read input_csv, drop any bbox overlapping > iou_thresh with a larger box,
    and save the survivors to output_csv.
    """
    df = pd.read_csv(input_csv)
    
    # 1) parse bboxes and compute area
    df["__bbox"] = df["rect_bbox"].apply(_parse_bbox)
    df["__area"] = df["__bbox"].apply(lambda b: max(0,(b[2]-b[0])) * max(0,(b[3]-b[1])))
    
    # 2) sort indices by descending area
    idxs = sorted(df.index, key=lambda i: df.at[i, "__area"], reverse=True)
    
    keep = []
    suppressed = set()
    
    # 3) greedy suppression
    for i in idxs:
        if i in suppressed:
            continue
        keep.append(i)
        bi = df.at[i, "__bbox"]
        
        for j in idxs:
            if j == i or j in suppressed:
                continue
            if _bbox_iou(bi, df.at[j, "__bbox"]) >= iou_thresh:
                suppressed.add(j)
    
    # 4) write out only the kept rows
    out = df.loc[keep].copy()
    out = out.drop(columns=["__bbox", "__area"])
    out.to_csv(output_csv, index=False)
    
    print(f"""
Suppressed overlapping bboxes (IoU ≥ {iou_thresh*100:.0f}%):
  - original count: {len(df)}
  - kept count:     {len(keep)}
  - dropped count:  {len(df) - len(keep)}
Saved result to: {output_csv}
""")


In [49]:
# 1) Suppress on axis-aligned boxes (fast):
suppress_by_graph(
    input_csv="annotated_output_test_1/final_rect_merged.csv",
    output_csv="annotated_output_test_1/final_bbox_sup.csv",
    iou_thresh=.9
)

# # 2) Suppress on true polygon IoU (slower, but more precise):
# global_suppression(
#     input_csv="final_rect_merged.csv",
#     output_csv="final_mask_sup.csv",
#     iou_thresh=0.9,     # you can tweak this to 0.8, 0.95, …
#     mode="mask"
# )



Graph-based suppression (IoU ≥ 90%):
  original count: 82
  components:     61
  kept largest:   61
  dropped total:  21
Result written to: annotated_output_test_1/final_bbox_sup.csv

